# Verificación de acceso a AWS con `boto3` sin credenciales en el código

Este notebook utiliza únicamente el nombre del perfil local (`tesis`) configurado en `~/.aws/`.
Nunca se escriben llaves ni identificadores de cuenta en el código o en sus salidas.

In [1]:
import boto3

# Único dato de sesión permitido en el código: el nombre del perfil en ~/.aws/
PROFILE_NAME = "tesis"
BUCKET_NAME = "lab19-jorge-prueba-20261116"

# Creamos una Session explícita para no depender del perfil default ni de variables de entorno
session = boto3.Session(profile_name=PROFILE_NAME)

# Verificamos con STS qué identidad respalda la sesión antes de tocar S3
identidad = session.client("sts").get_caller_identity()
arn_usuario = identidad["Arn"].split(":")[-1]  # Solo la parte final (ej. user/jorge-labs)
assert not arn_usuario.endswith("root"), "Nunca operes con la cuenta raíz"

print(f"Región activa: {session.region_name}")
print(f"Identidad (enmascarada): arn:aws:iam::************:{arn_usuario}")

Región activa: us-east-2
Identidad (enmascarada): arn:aws:iam::************:user/jorge-labs


In [2]:
# Reutilizamos la misma sesión para crear el cliente de Amazon S3
s3 = session.client("s3")

respuesta = s3.list_buckets()
buckets = [b["Name"] for b in respuesta.get("Buckets", [])]
print("Buckets encontrados:", buckets)
assert BUCKET_NAME in buckets, f"No se encontró el bucket esperado: {BUCKET_NAME}"

Buckets encontrados: ['lab19-jorge-prueba-20261116']


In [3]:
from pathlib import Path
import tempfile

objetos = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix="publico/")
claves = [obj["Key"] for obj in objetos.get("Contents", [])]
print("Objetos bajo publico/:", claves)

# Descargamos a un directorio temporal del sistema para no dejar artefactos sueltos en el repo Git
with tempfile.TemporaryDirectory() as tmpdir:
    destino = Path(tmpdir) / "archivo.txt"
    s3.download_file(BUCKET_NAME, "publico/archivo.txt", str(destino))
    print("Contenido descargado de publico/archivo.txt:")
    print(destino.read_text(encoding="utf-8"))

Objetos bajo publico/: ['publico/', 'publico/archivo.txt']
Contenido descargado de publico/archivo.txt:
archivo de prueba publico Lab 19
